# Notebook 02 — Seeing the Map (PCA)

> **Easiest way to run this: Google Colab — nothing to install.**
> Go to https://colab.research.google.com → **File > Upload notebook** → choose this file.
> Prefer your own computer? Lesson 1 shows the VS Code and local-Jupyter paths too.

We built 4-D word vectors and measured similarity — but we never *saw* the space. You can't
picture 4 directions at once. This notebook learns the one trick that works for *any* number
of directions, from our 4 up to the model's 384: **PCA**.

> **Analogy:** PCA is the angle you photograph a sculpture from to show the most detail. A
> sculpture has depth (3-D); a photo has 2. A bad angle hides what matters; a good angle
> reveals it. PCA finds the best angle automatically — the one where the data spreads out the
> most.

In [ ]:
# Installs the three tools this notebook needs (skip the wait if already installed).
%pip install -q numpy matplotlib scikit-learn
print("Ready.")

## Step 1 — Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

print("Imports ready.")

## Step 2 — The same ten words

Each notebook stands on its own, so we re-type the ten 4-D word vectors from Notebook 01.
The four directions are still `animal_ness`, `vehicle_ness`, `size`, `alive`. We also tag
each word with a category so the plot can colour them.

In [ ]:
#                       animal  vehicle  size   alive
word_vectors = {
    "cat":      [0.9,   0.0,   0.2,   0.9],
    "dog":      [0.9,   0.0,   0.3,   0.9],
    "lion":     [0.9,   0.0,   0.8,   0.9],
    "mouse":    [0.9,   0.0,   0.1,   0.9],
    "car":      [0.0,   0.9,   0.5,   0.7],
    "truck":    [0.0,   0.9,   0.9,   0.7],
    "bicycle":  [0.0,   0.8,   0.2,   0.5],
    "airplane": [0.0,   0.9,   1.0,   0.8],
    "tree":     [0.0,   0.0,   0.7,   0.6],
    "rock":     [0.0,   0.0,   0.5,   0.0],
}
category = {
    "cat": "animal", "dog": "animal", "lion": "animal", "mouse": "animal",
    "car": "vehicle", "truck": "vehicle", "bicycle": "vehicle", "airplane": "vehicle",
    "tree": "other", "rock": "other",
}
colour_map = {"animal": "tab:orange", "vehicle": "tab:blue", "other": "tab:green"}

print(f"{len(word_vectors)} words across {len(set(category.values()))} categories.")

## Step 3 — Approach A: just plot two of the directions

Because *we* named the directions, we can pick two and use them as the x- and y-axes. Let's
plot `animal_ness` (x) against `vehicle_ness` (y).

> This only works because we labelled the directions. With a real model's 384, nobody
> knows what each one means — so we'll need Approach B (PCA).

In [ ]:
words = list(word_vectors.keys())
xs = [word_vectors[w][0] for w in words]   # animal_ness
ys = [word_vectors[w][1] for w in words]   # vehicle_ness
colours = [colour_map[category[w]] for w in words]

plt.figure(figsize=(7, 5))
plt.scatter(xs, ys, c=colours, s=140, edgecolor="black")
for x, y, label in zip(xs, ys, words):
    plt.annotate(label, (x, y), xytext=(6, 4), textcoords="offset points", fontsize=11)
plt.xlabel("animal_ness  (direction 0)")
plt.ylabel("vehicle_ness  (direction 1)")
plt.title("Approach A — two hand-chosen directions")
plt.xlim(-0.1, 1.1); plt.ylim(-0.1, 1.1)
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

### The limitation

We threw away `size` and `alive`. This plot can't tell that `tree` is bigger than `rock`.
For 4 directions that's survivable; for 384 it's hopeless. Enter PCA.

## Step 4 — Approach B: PCA squashes every direction down to 2

PCA's job: look at all the data, find the single direction where points spread out the most
(call it **PC1**), then the next-best direction at right angles (**PC2**), and place every
point on those two. The result uses information from *all four* original directions.

**PC1 and PC2 aren't labelled** — PCA just finds the angle with the most contrast. You can
often squint and guess what they capture, but the model doesn't tell you.

> A PCA plot can come out **mirrored or rotated** on different machines — left/right is not
> meaningful. Only **closeness** (which points cluster together) is meaningful.

In [ ]:
# Activity: run PCA to squash the 4-D vectors down to 2-D.
matrix = np.array([word_vectors[w] for w in words])
print(f"Input shape: {matrix.shape}   (words, directions)")

pca = PCA(n_components=2)
pcs = pca.fit_transform(matrix)
print(f"Output shape: {pcs.shape}   (words, 2 principal components)")

var = pca.explained_variance_ratio_
print(f"\nPC1 captured {var[0] * 100:.0f}% of the spread.")
print(f"PC2 captured {var[1] * 100:.0f}% of the spread.")
print(f"Together: {sum(var) * 100:.0f}%   (the rest is lost squashing 4-D to 2-D)")

Now plot the 2-D result we just computed.

In [ ]:
# Activity: plot the PCA result (uses pcs and var from the cell above).
plt.figure(figsize=(8, 5.5))
plt.scatter(pcs[:, 0], pcs[:, 1], c=colours, s=140, edgecolor="black")
for (x, y), label in zip(pcs, words):
    plt.annotate(label, (x, y), xytext=(6, 4), textcoords="offset points", fontsize=11)
plt.xlabel(f"PC1  ({var[0] * 100:.0f}% of spread)")
plt.ylabel(f"PC2  ({var[1] * 100:.0f}% of spread)")
plt.title("Approach B — PCA from 4-D to 2-D (uses all directions)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

### What to notice
1. Animals cluster on one side, vehicles on the other, `tree` and `rock` off on their own —
   the same grouping as before, but now the axes are *found*, not hand-picked.
2. Squint: PC1 probably captures animal-vs-vehicle (where most variation lives); PC2 maybe
   size or alive.
3. The %s matter: PC1 usually captures the most, PC2 the next. Together ~80–90% here, the
   rest lost to flattening.

## Step 5 — Predict, then verify: add three words

We add three new words with hand-picked numbers, then re-run PCA on the bigger set. Before
running, guess where each lands — near which existing words?

- `scooter` — a small wheeled thing
- `whale` — a huge ocean animal
- `flower` — alive, small, not an animal

In [ ]:
#                                  animal  vehicle  size   alive
new_words = {
    "scooter":   [0.0,   0.7,   0.3,   0.5],
    "whale":     [0.9,   0.0,   1.0,   0.9],
    "flower":    [0.0,   0.0,   0.1,   0.7],
}
new_category = {"scooter": "vehicle", "whale": "animal", "flower": "other"}

all_vectors = {**word_vectors, **new_words}
all_categories = {**category, **new_category}
all_words = list(all_vectors.keys())
all_matrix = np.array([all_vectors[w] for w in all_words])
all_colours = [colour_map[all_categories[w]] for w in all_words]

pca2 = PCA(n_components=2)
pcs2 = pca2.fit_transform(all_matrix)

plt.figure(figsize=(9, 6))
for (x, y), word, c in zip(pcs2, all_words, all_colours):
    is_new = word in new_words
    plt.scatter(x, y, c=c, s=200 if is_new else 140,
                edgecolor="red" if is_new else "black",
                linewidths=2 if is_new else 1)
    plt.annotate(word, (x, y), xytext=(6, 4), textcoords="offset points",
                 fontsize=12, fontweight="bold" if is_new else "normal")
plt.xlabel(f"PC1  ({pca2.explained_variance_ratio_[0] * 100:.0f}% of spread)")
plt.ylabel(f"PC2  ({pca2.explained_variance_ratio_[1] * 100:.0f}% of spread)")
plt.title("PCA with three new words (red border)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

### Did your guesses land? — and recap

- `scooter` should sit in the vehicle cluster (near `bicycle`); `whale` in the animal
  cluster (pulled toward `lion` by size); `flower` near `tree`.
- Adding words makes PCA recompute from scratch, so exact positions shift — don't expect
  identical coordinates. Clusters stay; coordinates wander.

**Recap**
- PCA finds the angle where data is most spread out, and flattens onto it.
- Read a PCA plot by its clusters; the axis %s tell you how faithful the 2-D picture is.
- Pick-two-directions works when you named them; PCA works always.

**Next (Notebook 03):** enough hand-built numbers — we hand real sentences to a real model
and get back real 384-dimensional embeddings, then use these exact tools on them.